# 02 — Chasse OBJECTIVE au signal : peut-on extraire de l'information ?

> On reprend la base **comme un quant qui la découvre**, avec le mandat de **trouver du signal s'il
> existe** (pas de confirmer un a priori). **6 angles indépendants** ; pour chacun : *hypothèse → code →
> résultat → verdict (on garde / on lâche + pourquoi)*. Moteur importé (`data`, `prices`, `evaluate`).

⚠️ **Limites communes** (à garder en tête pour tous les angles) : **survivorship** (univers prix = tickers
encore cotés → CAR long-horizon = bornes hautes) ; **fenêtres chevauchantes** (les t « naïfs » sont
gonflés → on rapporte une version robuste **Newey-West** ou **clustering par membre**, c'est elle qui
dégonfle) ; **multiple testing** (des dizaines de coupes → ~1 « significatif » attendu par hasard).

In [1]:
import sys, os, warnings; warnings.filterwarnings('ignore')
for base in [os.path.expanduser('~/Downloads/Jupiter'), os.path.expanduser('~/Downloads/0. Jupiter')]:
    p = os.path.join(base, '00. S3S4 en cours')
    if os.path.exists(os.path.join(p, 'data.py')): S3 = p; break
sys.path.insert(0, S3)
import numpy as np, pandas as pd
from scipy.stats import spearmanr, ttest_ind
import data, prices, evaluate
df = data.load_transactions(); buys = data.buy_signals(df)
panel = prices.load_panel(list(df['ticker'].dropna().unique())); spy = prices.get_spy(); factors = prices.get_factors()
print(f"{len(df):,} transactions | {len(buys):,} achats | {panel.shape[1]} tickers en cache".replace(',', ' '))

113 675 transactions | 56 877 achats | 1188 tickers en cache


## Angle 1 — Information Coefficient : y a-t-il *de l'information* ?
**Hypothèse** : si les achats du Congrès portent de l'info, alors classer les titres par un **signal**
(nb d'acheteurs distincts le mois M) doit corréler (Spearman) avec leur **rendement futur**. Test
standard, indépendant de toute construction de portefeuille. On compare aussi **comptage vs montant $**.

In [2]:
mpx = panel.resample('ME').last(); fwd = {h: mpx.shift(-h)/mpx - 1.0 for h in (6, 12)}
bm = buys.copy(); bm['m'] = bm['filed'].dt.to_period('M').dt.to_timestamp('M')
sl = df[(df.op == 'sell') & df.ticker.notna()].copy(); sl['m'] = sl['filed'].dt.to_period('M').dt.to_timestamp('M')
buy_count = bm.groupby(['m', 'ticker'])['bioguide'].nunique()
net_usd = bm.groupby(['m', 'ticker'])['size_usd'].sum().subtract(sl.groupby(['m', 'ticker'])['size_usd'].sum(), fill_value=0)
def ic_series(sig, h):
    fr = fwd[h]; ics = []
    for m in mpx.index:
        if m in fr.index and m in sig.index.get_level_values(0):
            sv = sig.xs(m, level=0); rv = fr.loc[m].dropna(); j = sv.index.intersection(rv.index)
            if len(j) >= 8:
                ic = spearmanr(sv.loc[j], rv.loc[j]).correlation
                if not np.isnan(ic): ics.append(ic)
    return np.array(ics)
print('signal      horizon   IC      t_iid   t_NW   n_mois')
for name, sig in [('buy_count (breadth)', buy_count), ('net_usd ($ signe)', net_usd)]:
    for h in (6, 12):
        ic = ic_series(sig, h)
        print(f'{name:20s} {h:2d}m   {ic.mean():+.4f}  {ic.mean()/(ic.std(ddof=1)/len(ic)**.5):+.2f}   {evaluate.nw_tstat(ic, h):+.2f}   {len(ic)}')
# traduction économique : spread breadth (>=2 acheteurs - 1 acheteur) a 6 mois
sp = []
for m in mpx.index:
    if m in fwd[6].index and m in buy_count.index.get_level_values(0):
        sv = buy_count.xs(m, level=0); rv = fwd[6].loc[m].dropna(); j = sv.index.intersection(rv.index)
        if len(j) >= 25:
            sv, rv = sv.loc[j], rv.loc[j]; hi, lo = rv[sv >= 2], rv[sv == 1]
            if len(hi) >= 3 and len(lo) >= 3: sp.append(hi.mean() - lo.mean())
sp = np.array(sp)
print(f'\nSpread breadth (>=2 vs 1 acheteur), 6m, annualise : {sp.mean()*2*100:+.2f}%  t={sp.mean()/(sp.std(ddof=1)/len(sp)**.5):.2f}')

signal      horizon   IC      t_iid   t_NW   n_mois
buy_count (breadth)   6m   +0.0191  +2.30   +1.95   144
buy_count (breadth)  12m   +0.0200  +2.55   +2.86   138


net_usd ($ signe)     6m   +0.0081  +1.29   +1.27   144
net_usd ($ signe)    12m   +0.0010  +0.15   +0.16   138

Spread breadth (>=2 vs 1 acheteur), 6m, annualise : +1.88%  t=1.88


**Résultat** : le **nb d'acheteurs distincts** (breadth) a un IC ≈ **+0,02**, t_NW ~2-2,9 — *positif et
non nul*. Le **montant $** ne marche pas (t<1,3). Le spread breadth annualisé ≈ **+1,9 %, t≈1,9**.
**→ ON GARDE (comme INFORMATION)** : il y a un signal réel, faible, dans le *comptage* d'acheteurs — ni
beta, ni bruit. Mais minuscule et (on le verra) instable → à la limite de l'exploitable.

## Angle 2 — Event-study : les achats battent-ils le marché (et l'info existe-t-elle *avant* la divulgation) ?
**Hypothèse** : si conviction = information, les **gros** achats devraient surperformer. Test : rendement
anormal vs SPY (`evaluate.car_event`) à 252 j, par taille. **Placebo** : les grosses *ventes* devraient,
elles, *sous*-performer si c'est directionnel.

In [3]:
def car_bucket(sub, hdays=252):
    out = []
    for t, d in zip(sub['ticker'], sub['filed']):
        if t in panel.columns:
            c = evaluate.car_event(panel[t], spy, d, hdays)
            if c is not None: out.append(c)
    a = np.array(out); return a
def line(name, a):
    t = a.mean()/(a.std(ddof=1)/len(a)**.5) if len(a) > 2 else np.nan
    print(f'{name:28s} n={len(a):6d}  moy={a.mean()*100:+6.2f}%  med={np.median(a)*100:+6.2f}%  %>0={100*(a>0).mean():4.0f}%  t={t:+.2f}')
sells = df[(df.op == 'sell') & df.ticker.notna()]
print('--- ACHATS, rendement anormal 12 mois vs SPY ---')
line('tous les achats', car_bucket(buys))
line('achats >= 50k$', car_bucket(buys[buys.size_usd >= 50000]))
line('achats >= 250k$', car_bucket(buys[buys.size_usd >= 250000]))
print('--- PLACEBO : grosses VENTES (devraient sous-performer si directionnel) ---')
line('ventes >= 50k$', car_bucket(sells[sells.size_usd >= 50000]))
print('--- ACHATS >=50k par regime ---')
for lo, hi, lab in [(2014, 2018, '2014-2018'), (2019, 2021, '2019-2021'), (2022, 2026, '2022-2026')]:
    s = buys[(buys.size_usd >= 50000) & (buys.filed.dt.year.between(lo, hi))]
    line(lab, car_bucket(s))

--- ACHATS, rendement anormal 12 mois vs SPY ---


tous les achats              n= 39011  moy= +0.95%  med= -3.38%  %>0=  44%  t=+3.29
achats >= 50k$               n=  3256  moy= +2.69%  med= -1.10%  %>0=  48%  t=+3.29
achats >= 250k$              n=   360  moy= +3.40%  med= +0.53%  %>0=  51%  t=+1.93
--- PLACEBO : grosses VENTES (devraient sous-performer si directionnel) ---


ventes >= 50k$               n=  4008  moy= +2.34%  med= -1.84%  %>0=  47%  t=+2.79
--- ACHATS >=50k par regime ---
2014-2018                    n=  1175  moy= +0.74%  med= -2.35%  %>0=  45%  t=+0.48
2019-2021                    n=  1195  moy= +6.19%  med= +2.80%  %>0=  54%  t=+5.28
2022-2026                    n=   886  moy= +0.56%  med= -3.63%  %>0=  44%  t=+0.37


**Résultat** : agrégé = **rien** (médiane négative). Les **gros achats ≥250k** surperforment (~+4 %)
— **mais les grosses ventes AUSSI** (~+2,5 %) ⇒ c'est un **tilt de style** (large-cap croissance qui bat
le SPX quel que soit le sens), pas de l'info directionnelle. Et l'effet est **concentré sur 2019-2021**.
**→ ON LÂCHE** : l'asymétrie pré-divulgation existe faiblement mais l'« edge taille » est du beta + un
seul régime ; l'alpha FF-Carhart calendar-time n'est pas significatif (cf. notebook 01).

## Angle 3 — Long-short market-neutral : un alpha *caché par le beta* ?
**Hypothèse** : le long-only est dominé par β≈0,9 ; un portefeuille **long achats − short ventes**
neutralise le marché et isolerait un éventuel alpha.

In [4]:
ret = panel.pct_change().fillna(0.0).values
idx = panel.index; tpos = {t: i for i, t in enumerate(panel.columns)}; D, N = ret.shape
def side_weights(sub, hold=63):
    # poids quotidien EW des positions ouvertes (entree filed+1, detention `hold` j) — vectorise (deltas+cumsum)
    sub = sub[sub.ticker.isin(tpos)]
    ei = idx.searchsorted(sub['filed'].values + np.timedelta64(1, 'D'))
    ti = sub['ticker'].map(tpos).values
    deltas = np.zeros((D + 1, N)); ok = ei < D
    np.add.at(deltas, (ei[ok], ti[ok]), 1.0)
    np.add.at(deltas, (np.clip(ei + hold, 0, D)[ok], ti[ok]), -1.0)
    held = np.cumsum(deltas[:D], axis=0); held[held < 0] = 0
    tot = held.sum(axis=1, keepdims=True)
    return np.divide(held, tot, out=np.zeros_like(held), where=tot > 0)
sells = df[(df.op == 'sell') & df.ticker.notna()]
for hold, lab in [(21, '1m'), (63, '3m'), (126, '6m')]:
    w = side_weights(buys, hold) - side_weights(sells, hold)              # long achats - short ventes
    w_prev = np.vstack([np.zeros((1, N)), w[:-1]])
    ls = pd.Series((w_prev * ret).sum(axis=1), index=idx)                 # rendement quotidien
    fa = evaluate.factor_alpha(ls, factors)
    print(f'long-short {lab:3s} : alpha_factoriel={fa["alpha_annuel"]*100:+.2f}%/an  t={fa["alpha_t"]:+.2f}  beta_mkt={fa["beta_marche"]:+.3f}')

long-short 1m  : alpha_factoriel=-0.70%/an  t=-0.49  beta_mkt=-0.033
long-short 3m  : alpha_factoriel=-1.16%/an  t=-1.22  beta_mkt=-0.025
long-short 6m  : alpha_factoriel=-0.90%/an  t=-1.25  beta_mkt=-0.021


**Résultat** : alpha market-neutral **non significatif** (|t|<1,4) à 1/3/6 mois, **β≈0** (bien neutre).
**→ ON LÂCHE** : retirer le beta ne révèle **aucun** alpha. L'hypothèse « le beta cache l'alpha » est
**rejetée**.

## Angle 4 — Alignement commission ↔ secteur : la poche d'edge la plus citée
**Hypothèse** : un membre d'une commission achetant **son** secteur (Armed Services/Foreign Affairs →
défense=Industrials, Energy→Energy, Health→Health Care, Financial Services→Financials) a un avantage.
**Test décisif** : comparer brut, *puis* en **résidualisant par secteur×année** (retirer le beta sectoriel).

In [5]:
import glob
fin = pd.concat([pd.read_csv(f, dtype=str) for f in
                 glob.glob(os.path.join(os.path.dirname(S3), 'data/house/tables/*/06_house_*_FINAL.csv')) +
                 glob.glob(os.path.join(os.path.dirname(S3), 'data/senate/*/06_senate_*_FINAL.csv'))], ignore_index=True)
sec = fin.dropna(subset=['ticker', 'sector_gics']).groupby('ticker')['sector_gics'].agg(lambda s: s.value_counts().index[0])
JUR = {'Armed Services': 'Industrials', 'Foreign Affairs': 'Industrials', 'Foreign Relations': 'Industrials',
       'Homeland Security': 'Industrials', 'Energy': 'Energy', 'Natural Resources': 'Energy',
       'Health': 'Health Care', 'Ways and Means': 'Health Care', 'Financial Services': 'Financials', 'Banking': 'Financials'}
com = fin.dropna(subset=['bioguide_id', 'committee_membership']).groupby('bioguide_id')['committee_membership'].agg(lambda s: ' ; '.join(s.dropna().unique()))
def jur_sectors(bio):
    txt = com.get(bio, '') or ''
    return {v for k, v in JUR.items() if k in txt}
b = buys[buys.ticker.isin(sec.index) & buys.ticker.isin(panel.columns)].copy()
b['secteur'] = b['ticker'].map(sec); b['annee'] = b['filed'].dt.year
b['aligne'] = [s in jur_sectors(bio) for bio, s in zip(b['bioguide'], b['secteur'])]
b['car12'] = [evaluate.car_event(panel[t], spy, d, 252) for t, d in zip(b['ticker'], b['filed'])]
b = b.dropna(subset=['car12'])
al, na = b[b.aligne], b[~b.aligne]
print(f'BRUT @12m : aligne {al.car12.mean()*100:+.2f}% (n={len(al)}) vs non {na.car12.mean()*100:+.2f}% (n={len(na)}) | Welch t={ttest_ind(al.car12, na.car12, equal_var=False).statistic:+.2f}')
b['resid'] = b['car12'] - b.groupby(['secteur', 'annee'])['car12'].transform('mean')   # retire le beta secteur x annee
al, na = b[b.aligne], b[~b.aligne]
print(f'CONTROLE secteur x annee : aligne {al.resid.mean()*100:+.2f}% vs non {na.resid.mean()*100:+.2f}% | Welch t={ttest_ind(al.resid, na.resid, equal_var=False).statistic:+.2f}')
top = b[b.aligne & (b.secteur == 'Industrials')]['name'].value_counts()
print(f'Concentration (alignes Industrials) : top2 = {top.head(2).sum()/top.sum()*100:.0f}% ({list(top.head(2).index)})')

BRUT @12m : aligne +3.10% (n=3221) vs non +1.02% (n=33929) | Welch t=+2.70
CONTROLE secteur x annee : aligne -0.44% vs non +0.04% | Welch t=-0.67
Concentration (alignes Industrials) : top2 = 90% (['Ro Khanna', 'Michael T. McCaul'])


**Résultat** : en **brut**, les achats « de juridiction » battent nettement (+2-3 pts, t≈3). **Mais**
à **secteur×année constant**, l'effet **disparaît / s'inverse** : c'était du **beta défense (Industrials)**
qui a rallié 2022-2024. Et ~86 % des alignés viennent de **2 personnes** (Khanna+McCaul).
**→ ON LÂCHE** : la thèse d'edge la plus citée est un **artefact** (beta sectoriel + concentration).

## Angle 5 — Caractéristiques de trade + le test décisif (clustering par membre)
**Hypothèse** : certaines coupes (grosse taille, House, divulgation rapide) portent du signal.
**Test décisif** : un même membre génère des centaines de trades **corrélés** ; le t honnête se calcule
en **clusterisant par membre** (moyenne par membre), pas en comptant chaque trade comme indépendant.

In [6]:
b = buys[buys.ticker.isin(panel.columns)].copy()
b['car12'] = [evaluate.car_event(panel[t], spy, d, 252) for t, d in zip(b['ticker'], b['filed'])]
b = b.dropna(subset=['car12']); b['delai'] = (b['filed'] - b['traded']).dt.days
def naive_vs_clustered(sub, lab):
    x = sub['car12']
    t_naif = x.mean()/(x.std(ddof=1)/len(x)**.5)
    per_member = sub.groupby('bioguide')['car12'].mean()              # 1 obs / membre
    t_clu = per_member.mean()/(per_member.std(ddof=1)/len(per_member)**.5)
    print(f'{lab:24s} moy={x.mean()*100:+6.2f}%  n={len(x):5d}  t_naif={t_naif:+.2f}  ->  t_clusterise={t_clu:+.2f}  ({len(per_member)} membres)')
naive_vs_clustered(b[b.size_usd >= 250000], 'taille >=250k')
naive_vs_clustered(b[(b.size_usd >= 250000) & (b.chamber == 'house')], 'House & >=250k')
naive_vs_clustered(b[b.delai <= 10], 'divulgation <=10j')
naive_vs_clustered(b[b.size_usd < 50000], '(reference: <50k)')

taille >=250k            moy= +3.40%  n=  360  t_naif=+1.93  ->  t_clusterise=+0.25  (34 membres)
House & >=250k           moy= +5.02%  n=  322  t_naif=+2.69  ->  t_clusterise=+0.84  (24 membres)
divulgation <=10j        moy= +3.00%  n= 4300  t_naif=+3.18  ->  t_clusterise=+1.16  (135 membres)
(reference: <50k)        moy= +0.80%  n=35754  t_naif=+2.59  ->  t_clusterise=+0.87  (210 membres)


**Résultat** : taille ≥250k, House>250k, divulgation rapide ont un **t naïf > 2,5**… qui **retombe
sous |t|=2 en clusterisant par membre**. Le « signal » est porté par une **poignée d'individus**
(Mark Green, Khanna, Perdue), pas par une propriété généralisable du *type* de trade.
**→ ON LÂCHE** comme règle (ce serait sélectionner des *membres*, pas des trades — et la sélection par
perf passée ne persiste pas, cf. littérature).

## Angle 6 — ML attrape-tout : une structure prédictive *quelconque* ?
**Hypothèse** : si une structure (même non-linéaire) existe, un gradient boosting bien **validé dans le
temps** (train 2014-2020 / test OOS 2021+) la trouvera. Cible = signe du rendement anormal 6 mois.

In [7]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
b = buys[buys.ticker.isin(panel.columns)].copy()
b['car6'] = [evaluate.car_event(panel[t], spy, d, 126) for t, d in zip(b['ticker'], b['filed'])]
b = b.dropna(subset=['car6']); b['y'] = (b['car6'] > 0).astype(int)
b['annee'] = b['filed'].dt.year; b['mois'] = b['filed'].dt.month; b['delai'] = (b['filed'] - b['traded']).dt.days
b['ticker_code'] = b['ticker'].astype('category').cat.codes
b['party_code'] = b['party'].astype('category').cat.codes
b['chamber_code'] = (b['chamber'] == 'house').astype(int)
b['member_code'] = b['bioguide'].astype('category').cat.codes
feats = ['size_usd', 'delai', 'mois', 'party_code', 'chamber_code', 'ticker_code', 'member_code']
tr, te = b[b.annee <= 2020], b[b.annee >= 2021]
m = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=0).fit(tr[feats], tr['y'])
auc_tr = roc_auc_score(tr['y'], m.predict_proba(tr[feats])[:, 1])
auc_te = roc_auc_score(te['y'], m.predict_proba(te[feats])[:, 1])
base = max(te['y'].mean(), 1 - te['y'].mean())
print(f'AUC  train={auc_tr:.3f}   OOS(2021+)={auc_te:.3f}   (0,5 = hasard)')
print(f'accuracy OOS={(m.predict(te[feats]) == te["y"]).mean():.3f}  vs baseline classe-majoritaire={base:.3f}')
p = m.predict_proba(te[feats])[:, 1]; q4 = te[p >= np.quantile(p, 0.75)]
print(f'quartile haute-confiance OOS : rendement anormal moy={q4.car6.mean()*100:+.2f}%  med={q4.car6.median()*100:+.2f}%  win={100*(q4.car6>0).mean():.0f}%')

AUC  train=0.808   OOS(2021+)=0.511   (0,5 = hasard)
accuracy OOS=0.514  vs baseline classe-majoritaire=0.559
quartile haute-confiance OOS : rendement anormal moy=-1.45%  med=-2.60%  win=44%


**Résultat** : AUC **OOS ≈ 0,5** alors que train ≈ 0,7 (**sur-apprentissage**), accuracy sous la
baseline ; le quartile « haute confiance » a une **médiane négative**.
**→ ON LÂCHE** : aucune structure prédictive exploitable ; le gain moyen vient d'une queue, pas du trade
typique.

## Synthèse de la chasse au signal

| Angle | Verdict |
|---|---|
| **IC (breadth)** | ✅ **on garde comme INFO** : nb d'acheteurs distincts, IC≈0,02 (ni beta ni bruit) — mais minuscule & instable |
| Event-study taille | ❌ tilt de style (les ventes battent aussi) + 1 seul régime |
| Long-short | ❌ aucun alpha market-neutral (β≈0) |
| Commission↔secteur | ❌ artefact (beta défense + 2 traders) |
| Caractéristiques | ❌ s'effondre au clustering par membre (poignée d'individus) |
| ML | ❌ AUC OOS ≈ 0,5 (sur-apprentissage) |

**Conclusion réconciliée (objective, pas a priori)** : il existe une **information faible mais réelle** —
la *breadth* d'achat — qui n'est **ni du beta, ni du bruit**. Mais elle est **sous le seuil
d'exploitabilité** net de coûts : IC minuscule × **instabilité de régime** × **survivorship** × **coûts**
× **concentration** sur quelques membres. C'est pourquoi un produit **data** (Quiver) a de la valeur (il y
a une info à faire remonter) **sans** qu'une stratégie suivable batte le marché net de coûts. Le verdict
net du portefeuille est dans **`01_backtest.ipynb`**.